# 🎛️ EnerGIS Interactive Dashboard

Interaktives Dashboard zur Visualisierung gespeicherter EnerGIS Optimierungsergebnisse.

## Features

- 📊 **Overview**: KPIs und Zusammenfassung auf einen Blick
- 📈 **Zeitreihen**: Interaktive Plots mit Komponenten-Auswahl und Zoom
- 💰 **Kosten**: Detaillierte Kostenanalyse mit interaktiven Tabellen
- 🏭 **Anlagen-Design**: Kapazitäten und Auslegung
- 🔀 **Vergleich**: PF vs RH/MPC Vergleich

## Verwendung

1. Führe die Setup-Zellen aus
2. Wähle einen gespeicherten Workflow aus
3. Das Dashboard wird automatisch geladen
4. Interagiere mit den Plots und Tabellen

---

## 1. Setup & Dependencies

In [ ]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# Auto-Setup mit notebook_helpers
from energis.io.notebook_helpers import setup_notebook_environment

PROJECT_ROOT = setup_notebook_environment()
print("\n✅ Setup abgeschlossen")

In [ ]:
# Prüfe Dashboard-Dependencies
missing_deps = []

try:
    import panel as pn
    print(f"✅ Panel {pn.__version__} verfügbar")
except ImportError:
    print("❌ Panel nicht gefunden")
    missing_deps.append("panel")

try:
    import holoviews as hv
    print(f"✅ Holoviews {hv.__version__} verfügbar")
except ImportError:
    print("❌ Holoviews nicht gefunden")
    missing_deps.append("holoviews")

try:
    import plotly
    print(f"✅ Plotly {plotly.__version__} verfügbar")
except ImportError:
    print("❌ Plotly nicht gefunden")
    missing_deps.append("plotly")

if missing_deps:
    print(f"\n⚠️  Fehlende Dependencies: {', '.join(missing_deps)}")
    print(f"\n📦 Installation:")
    print(f"   pip install {' '.join(missing_deps)} bokeh")
else:
    print("\n✅ Alle Dependencies verfügbar!")
    # Load both plotly (for charts) and tabulator (for interactive tables)
    pn.extension('plotly', 'tabulator', sizing_mode='stretch_width')

In [ ]:
# Imports
from energis.io.notebook_helpers import (
    list_saved_workflows,
    load_workflow_from_saved,
    create_and_display_dashboard
)

print("✅ Imports erfolgreich")

## 2. Gespeicherte Workflows

Liste aller verfügbaren Workflows aus `saved_workflows/`.

In [ ]:
# Liste alle gespeicherten Workflows
workflows = list_saved_workflows(sort_by="date")

if not workflows:
    print("⚠️  Keine gespeicherten Workflows gefunden!")
    print("\n💡 Workflows erstellen:")
    print("   1. Öffne runner.ipynb oder scenario_studio.ipynb")
    print("   2. Führe eine Optimierung aus")
    print("   3. Der Workflow wird automatisch in saved_workflows/ gespeichert")
else:
    print(f"📦 {len(workflows)} gespeicherte Workflows gefunden:\n")
    print(f"{'#':<4} {'Name':<35} {'Datum':<20} {'Kosten [EUR]':<15} {'Steps'}")
    print("-" * 90)
    
    for i, wf in enumerate(workflows, 1):
        name = wf['name'][:33] + ".." if len(wf['name']) > 35 else wf['name']
        date_str = wf.get('date_str', '')[:19] if wf.get('date_str') else 'N/A'
        costs = f"{wf['costs']:,.0f}" if wf['costs'] > 0 else 'N/A'
        steps = ' → '.join(wf['steps']) if wf['steps'] else 'N/A'
        
        print(f"{i:<4} {name:<35} {date_str:<20} {costs:<15} {steps}")

## 3. Workflow auswählen

Wähle einen Workflow aus dem Dropdown-Menü.

In [ ]:
if workflows:
    # Erstelle Workflow-Auswahl
    import panel as pn
    
    workflow_selector = pn.widgets.Select(
        name='🗂️ Workflow auswählen',
        options={wf['name']: str(wf['path']) for wf in workflows},
        value=str(workflows[0]['path']),  # Default: neuester Workflow
        sizing_mode='stretch_width'
    )
    
    # Zeige Selector
    workflow_selector
else:
    print("⚠️  Keine Workflows zum Auswählen verfügbar")

## 4. Workflow laden

Lädt den ausgewählten Workflow aus `saved_workflows/`.

In [ ]:
if workflows:
    # Lade ausgewählten Workflow
    selected_path = workflow_selector.value
    
    print(f"📂 Lade Workflow...")
    print(f"   Pfad: {selected_path}\n")
    
    workflow = load_workflow_from_saved(selected_path, verbose=True)
    
    # Zeige kurze Info
    print("\n📊 Workflow-Info:")
    primary_result = workflow.rh_result or workflow.mpc_result or workflow.pf_result
    if primary_result:
        print(f"   Zeitraum: {primary_result.table.index[0]} bis {primary_result.table.index[-1]}")
        if primary_result.costs:
            total_cost = primary_result.costs.get('objective.OBJ_value_EUR', 0.0)
            print(f"   Gesamtkosten: {total_cost:,.0f} EUR")
    
    workflow_loaded = True
else:
    print("⚠️  Kein Workflow zum Laden verfügbar")
    workflow_loaded = False

## 5. Dashboard erstellen

Erstellt und zeigt das interaktive Dashboard mit allen Visualisierungen.

In [ ]:
if workflow_loaded and workflow:
    # Finde Workflow-Name für Titel
    workflow_name = "Unknown"
    for wf in workflows:
        if str(wf['path']) == selected_path:
            workflow_name = wf['name']
            break
    
    # Dashboard erstellen
    dashboard = create_and_display_dashboard(
        workflow,
        title=f"📊 {workflow_name}"
    )
    
    print("\n💡 Dashboard-Features:")
    print("   • Navigiere zwischen Tabs (Overview, Zeitreihen, Kosten, Design, Vergleich)")
    print("   • Im Zeitreihen-Tab: Wähle Komponenten und Zeitbereich")
    print("   • Plots sind interaktiv: Zoom, Pan, Hover für Details")
    print("   • Kosten-Tabelle: Sortierung durch Klick auf Spalten")
    print("\n🌐 Als Webapp exportieren:")
    print("   panel serve interactive_dashboard.ipynb --show")
    
    # Dashboard anzeigen
    dashboard
    
else:
    print("⚠️  Dashboard kann nicht erstellt werden - kein Workflow geladen")

---

## 📚 Weitere Optionen

### Anderen Workflow laden

Um einen anderen Workflow zu laden:
1. Gehe zurück zu Zelle 4 (Workflow auswählen)
2. Wähle einen anderen Workflow aus dem Dropdown
3. Führe die Zellen 5 und 6 erneut aus

### Dashboard als Webapp

```bash
# Terminal:
panel serve interactive_dashboard.ipynb --show

# Oder auf spezifischem Port:
panel serve interactive_dashboard.ipynb --port 5006 --show
```

### Dashboard-Komponenten einzeln verwenden

```python
from energis.io.dashboard import EnerGISDashboard

dash_obj = EnerGISDashboard(workflow, "Custom View")

# Einzelne Tabs:
# dash_obj._create_overview_tab()
# dash_obj._create_timeseries_tab()
# dash_obj._create_costs_tab()
# dash_obj._create_design_tab()
# dash_obj._create_comparison_tab()
```

---

## 🐛 Troubleshooting

**Dashboard wird nicht angezeigt:**
- Stelle sicher, dass Panel, Plotly und Holoviews installiert sind
- In JupyterLab: `jupyter labextension install @pyviz/jupyterlab_pyviz`
- Neustart des Kernels kann helfen

**Plots sind leer:**
- Überprüfe ob die Optimierung erfolgreich war
- Schaue ob Daten in workflow.pf_result oder workflow.rh_result vorhanden sind

**Keine gespeicherten Workflows:**
- Führe zuerst runner.ipynb oder scenario_studio.ipynb aus
- Workflows werden automatisch in `saved_workflows/` gespeichert

**Webapp startet nicht:**
- Prüfe ob Port 5006 frei ist
- Versuche: `panel serve interactive_dashboard.ipynb --port 5007`

---

**Viel Erfolg mit dem Dashboard! 🚀**